# Работа №3. Генерация/стилизация видео
## Текст задания

Задача:
Создать видео нейросетью:
Покадровая обработка (ControlNet или Image-to-Image) или
Латентная интерполяция из текстовых промптов.
Видео ≥5 секунд, 15 fps.

Результаты:
Исходное видео (если есть).
Код (.ipynb).
Результирующее видео.
Описание идеи.

Критерии:
Оригинальная идея.
Качество и сложность исполнения.

Сдача:
Файлы в облако, ссылка в комментариях.
Финальный проект / Экзамен

## Выполнение задания
Попробуем покадрово обрабатывать исходное видео и стилизовать его. Мы берем обычное видео и превращаем его в анимированное полотно в стиле Ван Гога. Используем метод Image-to-Image на базе Stable Diffusion, чтобы сохранить контуры объектов, но полностью изменить текстуру

Сначала установим всё нужное. Нам понадобятся diffusers для нейронки и opencv для работы с кадрами видео. А видеоролик скачан из интернета длительностью 7 секунд

In [ ]:
# Ставим диффузоры и ускорители и библиотеку для работы с видео
!pip install -q diffusers transformers accelerate safeTensors
!pip install -q opencv-python

Подготовка кадров и модели
Тут мы прописываем логику: разбиваем видео на картинки, прогоняем через SD и склеиваем обратно.

In [14]:
import torch
import numpy as np
import cv2
import os
import subprocess
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5" # Будем использовать небольшую модель, чтобы хватило ресурсов в колабе
# Загружаем модель с отключенным фильтром
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to("cuda")

PROMPT = "Oil painting, impressionism style, thick brushstrokes, vibrant colors, masterpiece, canvas texture, Claude Monet style"
NEGATIVE_PROMPT = "cyberpunk, neon, robots, technology, futuristic, high tech, blur, low quality, distorted"
STRENGTH = 0.45  # Чем выше, тем сильнее меняется видео

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def process_video(input_path, output_path):
    if not os.path.exists('frames'):
        os.makedirs('frames')

    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 15 # Страховка, если OpenCV не определил FPS, а нам строго 15 нужно
    frame_count = 0
    new_size = (512, 512) # Потому что слишком большой формат исходного видео. Его очень долго ждать

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Обработка нейронкой
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        init_image = Image.fromarray(frame_rgb).resize(new_size)

        with torch.autocast("cuda"):
            result_img = pipe(prompt=PROMPT,
                              negative_prompt=NEGATIVE_PROMPT,
                              image=init_image,
                              strength=STRENGTH).images[0]
        frame_name = f"frames/{frame_count:04d}.png" # Сохраняем каждый кадр как картинку по формату
        result_img.save(frame_name)

        frame_count += 1
        if frame_count % 10 == 0:
            print(f"Готово кадров {frame_count}")

    cap.release()

    # Собираем картинки в MP4 H.264
    subprocess.call([
        'ffmpeg', '-y', '-i', 'frames/%04d.png',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-r', str(fps), output_path
    ])

In [16]:
# Путь к твоему видео
input_video = "/content/People_480.mp4"
output_video = "final_artwork.mp4"

process_video(input_video, output_video)

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

...

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

## Итог
Генерация видео по кадрам прошла успешно. Видео преобрело художественность и стало необычным :)
Однако, было трудно использовать и настраивать StableDiffusionImg2ImgPipeline... теперь умеею работать с весами моделей и параметром strength (контроль изменений).

Код выдает честные 15+ fps, всё по критериям

На некоторых примерах видео дергалось, это нормально для базового покадрового метода.
Сам ролик появляется в файлах, нужно его искать там